# Decision Trees
---

### Overview
- [Toy Example on Decision Boundaries](#Toy-Example-on-Decision-Boundaries)
- [Decision Tree for Diabetes Classification](#Decision-Tree-for-Diabetes-Classification)
  - [Data Understanding and Preprocessing](#Data-Understanding-and-Preprocessing)
  - [Training a Decision Tree (including a stopping criteria)](#Training-a-Decision-Tree-(including-a-stopping-criteria))
  - [Visualizing Trees](#Visualizing-Trees)
  - [Training a DT with stopping criteria of impurity decrease](#Training-a-DT-with-stopping-criteria-of-impurity-decrease)
  - [Building a DT with Post-Pruning](#Building-a-DT-with-Post-Pruning)
- [Regression Trees](#Regression-Trees)  

In this notebook we investigate decision trees for **classification** and **regression**.

Please note that `scikit-learn` has a **CART implementation only** (no ID3, no C4.5), so it cannot be used with categorical data in a straight-forward way. With scikit-learn, only solutions with one-hot-encoding OR OrdinalEncoder (treating categories as numerical data) could be used with categorical data -- both "hacks" might not be ideal. For that reason, a third party library called `chefboost` could be used. However, this library was developed by a single developer, has not been tested intensively, and also suffered from bugs not being fixed for a longer time in the past. Furthermore, the library does not follow the typcial class and method design from scikit-learn. For that reason, a small additional notebook how this library could be used is **[available here](MLEA_07_MAT_Chefboost-SUPPLEMENT.ipynb)**. Please note, that this `chefboost`-notebook is **additional** notebook (maybe it is helpful if you have to deal with categorical data in the future/in your project), but it is not relevant for the exam/the focus of this lecture. It is recommended to work through this material notebook first...

As always, first the imports:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn import tree # required for plot.
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error
from sklearn.model_selection import GridSearchCV

## Toy Example on Decision Boundaries

Let's start with an artificial 2D example similar to what we did in K-NN in order to investigate the decision boundary. Again, we create data with two classes, which is not linerarly separable:

In [ ]:
X, y = make_moons(n_samples=200, noise=0.4, random_state=1) 
# maybe you remember the make_moons function from the clustering notebook...
sc = MinMaxScaler()
X = sc.fit_transform(X) # normalize, not neccessary for DT, but doesn't harm..

plt.figure(figsize=(8, 4))
plt.plot(X[y.ravel()==0, 0], X[y.ravel()==0, 1], "bo", markeredgecolor="k")
plt.plot(X[y.ravel()==1, 0], X[y.ravel()==1, 1], "ro")

plt.title("2D Non-Linear Data Distribution", fontsize=16)
plt.xlabel("$x_1$", fontsize=14)
plt.ylabel("$x_2$", fontsize=14)
plt.show()

The same helper function as in previous notebooks is used again to plot the decision boundaries:

In [ ]:
def decision_boundary_class_colored(clf, X, plotDistanceFromHyperplane=False, colorBar=False):
    # Get the min and max value of feature x1
    x1min, x1max = X[:,0].min() - 0.1, X[:, 0].max() + 0.1
    
    # Get the min and max value of feature x2
    x2min, x2max = X[:,1].min() - 0.1, X[:, 1].max() + 0.1
    
    # Create the mesh grid
    x1s = np.linspace(x1min, x1max, 100)
    x2s = np.linspace(x2min, x2max, 100)
    x1, x2 = np.meshgrid(x1s, x2s)
    
    # Create pairs of new points from the grid
    X_new = np.c_[x1.ravel(), x2.ravel()]
    
    # Compute the class predictions for all new points
    y_pred = clf.predict(X_new).reshape(x1.shape)
    
    # Generate the contourf plot for the predictions
    plt.contourf(x1, x2, y_pred, cmap=plt.cm.seismic, alpha=0.2)
    
    if(plotDistanceFromHyperplane == True):
        # Compute the signed distance of a sample to the hyperplane for all new points
        y_decision = clf.decision_function(X_new).reshape(x1.shape)

        # Generate the contourf plot for the distance of all points from the hyperplane
        plt.contourf(x1, x2, y_decision, cmap=plt.cm.bwr, alpha=0.2)
    
    if(colorBar==True):
        plt.colorbar()  

Now we train a Decision Tree without any stopping criteria (t1) and one tree with a max depth of 3 levels (t2). We print the accuracies as well as the decision boundaries for both trees:

In [ ]:
# Split into train/test:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

t1 = DecisionTreeClassifier(random_state=0)
t2 = DecisionTreeClassifier(random_state=0, max_depth=3)
t1.fit(X_train, y_train)
t2.fit(X_train, y_train)

print('Test accuracy with full tree, levels='+str(t1.tree_.max_depth)+":", t1.score(X_test, y_test))
print('Test accuracy with levels=3:', t2.score(X_test, y_test))
    
# Show the decision boundary only along with training data
for m in [t1, t2]:
    plt.figure(figsize=(8,4))
    decision_boundary_class_colored(m, X_train,)
    plt.plot(X_train[y_train.ravel()==0, 0], X_train[y_train.ravel()==0, 1], "bo")
    plt.plot(X_train[y_train.ravel()==1, 0], X_train[y_train.ravel()==1, 1], "ro")
    plt.title(m , fontsize=16)
    plt.show()

It can be easily seen that the decision tree with a depth of 10 is clearly overfitted.

First, we want to plot the rules of the tree (now using the shallow tree):

In [ ]:
#r = export_text(t1, feature_names=['feature_1', 'feature_2'])
#print(r)
r = export_text(t2, feature_names=['feature_1', 'feature_2'])
print(r)

It is even possible to generate Python code for these decision rules as pointed out in [this](https://stackoverflow.com/questions/20224526/how-to-extract-the-decision-rules-from-scikit-learn-decision-tree) post. 

Additionally, the `tree_`object can be used to query single nodes in the tree, more information is available in the [documentation](https://scikit-learn.org/stable/auto_examples/tree/plot_unveil_tree_structure.html), without further use in this notebook.

Trees can also be visualized nicely, but let's do that with a real world example below.

## Decision Tree for Diabetes Classification

This time we use a dataset containing numerical values from Kaggle. The task is to predict whethere a person will suffer from diabetes based on medical measurements (see below). More info can be found [here](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database/). The "Outcome" is the target variable.

### Data Understanding and Preprocessing

First we load the data and perform some data understanding:

In [ ]:
dia = pd.read_csv("data/diabetes.csv")
dia.head()

In [ ]:
pd.DataFrame(dia.describe())

As we can see from the mean value of the outcome, the classes are not balanced!

In [ ]:
print('diabetes:', len(dia[(dia['Outcome']==1)])/len(dia))
print('no diabetes:', len(dia[(dia['Outcome']==0)])/len(dia))

Finally, we check for null values:

In [ ]:
dia.isna().sum()

Separate data into X and y and do a train/test split. Please note, that we did **not scale/normalize the data**, since this is not required for a DT:

In [ ]:
y = dia['Outcome']
X = dia.iloc[:,0:-1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X.columns # outcome not in X any more

### Training a Decision Tree (including a stopping criteria)

Firstly, let's create a tree and visualize it. To this end, we ignore the inbalance of the classes. Maybe it would be better to balance them (e.g. 50% - 50%) to avoid that the tree includes the class prior (especially when pruning)

In [ ]:
# Create Decision Tree
clf = DecisionTreeClassifier(random_state=0) # this is the classifier from scikit-learn, basically a CART
# Train/fit Decision Tree Classifer
clf = clf.fit(X_train,y_train)

# Predict the test dataset
y_pred = clf.predict(X_test)

# and check accuracy
print('accurary:', accuracy_score(y_test, y_pred))
print('f1:', f1_score(y_test, y_pred))

And plot the tree using the build-in method from `scikit-learn` -- please be patient, it takes some time:

In [ ]:
fig = plt.figure(figsize=(50,50))
text = tree.plot_tree(clf, 
                   feature_names=dia.columns.to_list(), 
                   filled=True)

plt.show()

# print(text)

As it can be observed, the tree is very deep and most likely overfitted. If you double-click the image, you can zoom in, but still not really see much...

Before we start now "guessing" a good value for *max_depth* or the *criterion*, let's use a systematic way (X-validation, compare lecture last week!):

In [ ]:
cls = DecisionTreeClassifier(random_state=0)

params = {'max_depth':np.arange(3,15),
#          'min_samples_leaf':[3,5,10,15,20],
#          'min_samples_split':[8,10,12,18,20,16],
          'criterion':['gini','entropy']}
gs = GridSearchCV(cls, params, scoring='accuracy', cv=10, verbose=3, n_jobs=-1)
gs.fit(X_train, y_train)

**Note**: It is also possible to use *min_samples_leaf* or *min_samples_split* (or even combinations) for hyperparameter tuning (c.f. [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)), but it often does not give you much better results. However, you can try it out!

In [ ]:
params_optimal = gs.best_params_

print("Best Score: %f" % gs.best_score_)
print("Optimal Hyperparameter Values: ", params_optimal)

In [ ]:
tree_best = DecisionTreeClassifier(random_state=0, criterion='entropy', max_depth=3) #, min_samples_leaf=20, min_samples_split=8)
tree_best.fit(X_train, y_train)
pred = tree_best.predict(X_test)

print('Test accuracy',accuracy_score(y_test, pred))

Not bad, for such a small tree a pretty good result! Finally, let's plot the tree and observe the decison rules:

### Visualizing Trees

In [ ]:
fig = plt.figure(figsize=(25,20))
text = tree.plot_tree(tree_best, 
                   feature_names=dia.columns.to_list(), 
                   filled=True)

plt.show()

It is very interesting to interpret the result: A small glucose level seems to predict no diabetes whereas a high glucose level in combination with a high BMI seems to predict the disease. Please also understand the `value`field in each node: it tells you how many items of the node belong to class 0 and class 1 each.

Another advanced visualization for decision trees is available in the package `dtreeviz`, please follow the installation in this order:
> ``conda install graphviz`` and there**after**
> ``conda install conda-forge::dtreeviz``
> 
> ~~(in case the latter doesn't work try ``pip install dtreeviz``)~~

It can be a bit tricky to get graphviz running, in case it does not, please comment out the following code. ``conda-forge`` is another repository, the packages from conda don't work right now.

In [ ]:
import dtreeviz

viz = dtreeviz.model(tree_best, X_train, y_train,
                feature_names=X_train.columns.to_list(),
                class_names=['Healthy','Diabetes'])

viz
viz.view(scale=1.5)  # if image is too small, set this to larger value!

Please ignore the warning above. It complains that we used a dataframe instead of a numpy array, whyever...

### Training a DT with stopping criteria of impurity decrease

Another idea of estimating a stopping criteria is observing by how much the impurity decreases using a split. If the impurity decrease is below a certain threshold *min_impurity_decrease*, say 10%, the split will not be performed. Using this as a stopping criteria can create more imbalanced trees as the method in the last subsection. However, it is maybe a bit more meaningful idea than just looking at the depth of the tree. So we can try it on our data -- since this is another hyperparameter to set, we will tune it again using CV:

In [ ]:
cls = DecisionTreeClassifier(random_state=0, criterion='gini') # 'entropy' does not make much sense due to the stopping criteria

params = {'min_impurity_decrease':np.arange(0,.5,.001)}
gs = GridSearchCV(cls, params, scoring='accuracy', cv=10, verbose=3, n_jobs=-1)
gs.fit(X_train, y_train)

In [ ]:
params_optimal = gs.best_params_

print("Best Score: %f" % gs.best_score_)
print("Optimal Hyperparameter Values: ", params_optimal)

In [ ]:
tree_best = DecisionTreeClassifier(criterion= 'gini', min_impurity_decrease=0.01)
tree_best.fit(X_train, y_train)
pred = tree_best.predict(X_test)

print('Test accuracy',accuracy_score(y_test, pred))

What a disappointing result! Although the best score above was promising!

In [ ]:
fig = plt.figure(figsize=(10,10))
text = tree.plot_tree(tree_best, 
                   feature_names=dia.columns.to_list(), 
                   filled=True)

plt.show()

Even if our tree looks reasonable (and actually to a certain extend quite similar to the one before), the accuracy on the test set is less convincing. So maybe we skip this approach for our dataset for now.

### Building a DT with Post-Pruning

Until now, we tuned a hyperparameter by building multiple trees and performing a CV. However, we did a lot of the computation multiple times, although it was not really required. The original CART algorithm suggests a different way: Why not building a complete tree and afterwards cutting it back ("pruning")? This is what can be achived in `scikit-learn` using the ccp_alpha parameter, whereas ccp means "cost complexity pruning".

In [ ]:
clf = DecisionTreeClassifier(random_state=0) # even if entropy gave better results on our data, we use gini (default) here
                                             # because of measuring impurity using gini was suggested by the CART developer, Mr. Breiman
                                             # [entropy would work nevertheless as well]
path = clf.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

Now we plot the alphas and the impurities:

In [ ]:
fig, ax = plt.subplots()
ax.plot(ccp_alphas[:-1], impurities[:-1], marker="o", drawstyle="steps-post")
ax.set_xlabel("effective alpha")
ax.set_ylabel("total impurity of leaves")
ax.set_title("Total Impurity vs effective alpha for training set")

plt.show()

Please try to understand this plot with what we discussed in the lecture (refer to the formula with the alpha): If alpha is zero, all leaves are pure (impurity is zero), but the tree is overfitted. If alpha is too big (here maybe 0.025), then the leaves are very impure, thus our model might be underfitted. So we need to find a good alpha balancing over- and underfitting (similar to how we determined a good $k$ in k-NN!

Please note that in this plot the last impurity and alpha was removed ([:-1]), because the tree with only one node is not very meaningful. However, we could now select an alpha based on an educated guess "in the middle", but there is a better way: Let's use (again) a CV to determine a good alpha! Again, we drop the last one from the list

In [ ]:
parameters = {'ccp_alpha':ccp_alphas[:-1].tolist()}
print(parameters)

In [ ]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), parameters, cv=10, refit=True)
gs.fit(X_train,y_train)
print(gs.best_params_)

Since the refit parameter was used, there is no need to train again the full tree with the determined alpha, it will be done automatically.

In [ ]:
tree_best = gs.best_estimator_

pred = tree_best.predict(X_test)
print('Test accuracy',accuracy_score(y_test, pred))

The best result so far, better as "pre-pruning"!. Finally, let's plot the tree:

In [ ]:
fig = plt.figure(figsize=(10,10))
text = tree.plot_tree(tree_best, 
                   feature_names=dia.columns.to_list(), 
                   filled=True)

plt.show()

Our result is very close to the pre-pruned tree using the *min_impurity_decrease*, only the last split is different. But no wonder, since the used techniques are very similar.

**Take home message**:<br>
Post-pruning often works better than pre-pruning. This is simply to the reason that the tree was initially build with the "full" training data set (10 folds) instead of 9 folds. Using impurity is preferred over using a tree "depth".

## Regression Trees

Finally, we want to use a Decision Tree for a regression task. Please remember that CART stands for Classification and **Regression** Trees. To this end, we have to use the `scikit-learn`'s `DecisionTreeRegressor` for that. But first, we import the data (this time by directly downloading it from the internet -- so you need to be online to run this ;) : 

In [ ]:
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep=r'\s+', skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2] # house price in $1000

# just for data understanding
column_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT']

df = pd.DataFrame(data, columns=column_names)
df['MEDV'] = (target*1000)

print (df.shape)
df.head()

More info about the dataset and the meaning of the columns can be found [here](http://www.cs.toronto.edu/~delve/data/boston/bostonDetail.html). The task is to **predict a house price (MEDV)**, last column based on other information collected by the U.S Census Service. This is a standard dataset used as an example often, but please note it is quite small (506 instances) and rather old/outdated. However, we use it in the following to perform a (small) regression task. 

First let's continue the data understanding:

In [ ]:
df.isna().sum()

In [ ]:
df.describe()

It can be seen how old the data is due to a max house price of $50,000 (actually its from the 70ies)

Let's split the data and train a Regression Tree:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=.2, random_state=0)

dtr = DecisionTreeRegressor(random_state=0)
dtr.fit(X_train, y_train)

pred = dtr.predict(X_test)

mse = mean_squared_error(y_test, pred)
print("MSE: ", mse)
print("RMSE: ", mse**(.5)) # no need to import math ;)

The RMSE can be interpreted nicely: It is the standard deviation of our predecitions. In other words: In 68% of the predictions, we are $5,450 or less off. 

In [ ]:
np.set_printoptions(suppress=True) # for nicer output of numpy, otherwise we get scientific notation like 6.7240e-02
print('Example test case #0:\n',X_test[0])
print('Real house value:', y_test[0]) # please remember, this is in $1000
print('Predicted house value:', pred[0]) 
print('Absolute error for #0:', abs(y_test[0]-pred[0]))
print('MSE for #0:',(y_test[0]-pred[0])**2)


In [ ]:
fig = plt.figure(figsize=(20,5))
x_ax = range(len(y_test))
plt.bar(x_ax, y_test, label="original", color='darkblue')
plt.bar(x_ax, pred, label="predicted" , edgecolor='red', color='white', alpha=0.5)
plt.title("Boston test and predicted data")
plt.xlabel('Test case')
plt.ylabel('House value')
plt.legend(loc='best',fancybox=True, shadow=True)
plt.grid(True)
plt.show()   

Please note that the order on the x-axis has no meaning, it is just the case number. 
It can be seen that the predictions are not bad, but two of the outliers are predicted very badly.

Visualize the decision tree:

In [ ]:
fig = plt.figure(figsize=(10,10))
text = tree.plot_tree(dtr, 
                   feature_names=df.columns.to_list(), 
                   filled=True)

plt.show()

Overfitted (as to be expected)... Let's do post-pruning!

In [ ]:
path = dtr.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

parameters = {'ccp_alpha':ccp_alphas[:-1].tolist()}

gs = GridSearchCV(DecisionTreeRegressor(random_state=42), parameters, cv=10, refit=True)
gs.fit(X_train,y_train)
print(gs.best_params_)

pred = gs.best_estimator_.predict(X_test)

mse = mean_squared_error(y_test, pred)
print("MSE: ", mse)
print("RMSE: ", mse**(.5))

In [ ]:
fig = plt.figure(figsize=(50,20))
text = tree.plot_tree(gs.best_estimator_, 
                   feature_names=df.columns.to_list(), 
                   filled=True)

plt.show()

# Click on it to zoom in

In [ ]:
fig = plt.figure(figsize=(20,5))
x_ax = range(len(y_test))
plt.bar(x_ax, y_test, label="original", color='darkblue')
plt.bar(x_ax, pred, label="predicted" , edgecolor='red', color='white', alpha=0.5)
plt.title("Boston test and predicted data")
plt.xlabel('Test case')
plt.ylabel('House value')
plt.legend(loc='best',fancybox=True, shadow=True)
plt.grid(True)
plt.show()   

As a result, our post-pruned tree is smaller, the MSE slightly better, but the two outliers (which probably mess with our MSE a lot), are still not predicted well.